[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/reciprocal/reciprocal.ipynb)

# Lifts and Reciprocal Diagrams

This example computes the reciprocal diagram of a planar truss from a lift of its mesh. Give every vertex of a planar triangle mesh a height and the mesh lifts to a polyhedral terrain. Read the terrain as an Airy stress function and its creases are the bars of a planar truss: how sharply the terrain bends across an edge is the force in that bar, tension where it bends convex and compression where it bends concave, and every interior vertex is in equilibrium by construction. The slopes of the terrain's faces are the vertices of a second mesh, the reciprocal diagram, with an edge across every bar, perpendicular to it and as long as the bar's force. Lifted onto the paraboloid, the reciprocal diagram is the Voronoi diagram and the forces are the cotangent weights.

The vertices are points, antivectors, intrinsic like all geometry, and a face of the terrain is a plane, a vector: their pairing, `plane & (point + up)`, with the point lifted to unit height, is the height there. Through three corners the plane is their heights times the reciprocal frame of the lifted corners, the corners met in pairs over all three met, `(b & c) / (a & b & c)`, linear in the heights. The reciprocal diagram's vertices are those planes with their height at the origin wedged away, `(plane ^ mv.z) * mv.z`, and a bar's stress is the jump in slope across it, paired with the corner across the bar.

In matrix notation the bar forces read as the entries of an equilibrium stress, a symmetric matrix $\Omega$ with zero row sums whose product with the matrix of vertex positions vanishes at every interior vertex, $\Omega P = 0$; the Maxwell–Cremona correspondence pairs every such stress with a lift.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga.algebras import VGA3D as ga
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.surfaces.reciprocal import render

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, mv = mesh_module.Mesh, mesh_module.mv
Scalar = ga.gatype.scalar()                         # Scalar+
# A face of the terrain is a plane, a vector; the vertices are points, antivectors.
Plane = ga.gatype.vector()                          # Vector+
Point = mesh_module.Point                           # Antivector-
# The point one unit above the origin: added to a point of the plane, it lifts it to unit height.
up = mv.z.dual()                                    # [] Point-


## 1. Slopes, stresses and the reciprocal diagram

A face of the terrain is a plane, and a plane over the plane of the mesh is one vector, its slope and its height at the origin, whose pairing with a point lifted to unit height, `plane & (point + up)`, is the height there. Two lifted corners meet in a vector that pairs to nothing with either; over the meet of all three it pairs to one with the third, so the plane through the three corners at their heights is their heights times those three vectors, the reciprocal frame of the corners: linear in the heights. Across an interior edge the slope jumps, and the jump pairs to nothing with the edge, since both planes agree along it; over the edge's length it is the bar's stress. The faces' slopes are the vertices of the reciprocal diagram, and the jumps its edges. On the paraboloid, at half the squared distance from the centre, each face's slope, read as a point by the metric, `slope.dual()`, is its circumcentre and each stress the edge's cotangent weight, half the cotangents of the two angles facing it; the bars across cocircular corners carry almost nothing.

In [ ]:
# The grid: its vertices per side across the square from -1 to 1, how far its interior vertices are nudged,
# and the seed of the nudges.
n_side = 5
jitter = 0.08
grid_seed = 11


In [ ]:
across = np.linspace(-1, 1, n_side)
coords = np.stack(np.meshgrid(across, across, indexing="ij"), axis=-1).reshape(-1, 2)   # [vertices, 2]
inside = (np.abs(coords) < 1).all(axis=-1)
coords[inside] += jitter * np.random.default_rng(grid_seed).normal(size=(inside.sum(), 2))
# Points of the plane, at height zero, and the mesh through them.
points = mv.antivector(np.concatenate([coords, np.zeros((len(coords), 1))], axis=-1)).field()   # Point-[vertices]
mesh = Mesh.triangulate(points)


In [ ]:
def slopes(mesh: Mesh, points: Point, heights: Scalar) -> Plane:
    """The plane of each face of the terrain, through its corners at their heights, pairs with a point lifted
    to unit height, `point + up`, to the height there: it holds the slope across the plane and the height at
    the origin. It is the corners' heights times the reciprocal frame of the corners lifted to unit height, the
    three planes pairing to one with their own corner and to nothing with the other two, so it is linear in the
    heights."""
    lifted = points + up                                                       # Point-[vertices]
    corners = mesh.incidence[2, 0] * lifted                                    # [corners] Point-[faces]
    tails, heads = mesh.incidence[1, 0] * lifted                               # Point-[edges], Point-[edges]
    # Two corners meet in a vector that pairs to nothing with either; over the meet of all three, it pairs
    # to one with the third.
    frame = (mesh.boundary[2, 1] * (tails & heads)) / (corners[0] & corners[1] & corners[2])   # [corners] Plane+[faces]
    # The heights at each face's corners, on the last axis.
    corner_heights = mesh.incidence[2, 0] * heights[..., None]                 # [..., corners] Scalar+[faces]
    return (frame * corner_heights).sum(axis=-1)                               # [...] Plane+[faces]


def stresses(mesh: Mesh, points: Point, heights: Scalar) -> Scalar:
    """The stress of each interior edge is the jump in slope between the faces on either side, measured towards
    the corner across the edge, over the edge's length; it is positive in tension, where the terrain bends
    convex. Which face is called left does not matter: swapping them turns both the jump and the side the
    corner lies on."""
    inner = mesh.interior_edges                                                # [inner]
    # The faces' planes summed back onto their edges, each signed as its face runs along the edge: the plane of
    # the face on the left less that of the face on the right.
    jump = (~mesh.d[1] * slopes(mesh, points, heights)).batch()[..., inner]    # [..., inner] Plane+
    # Each bar from its tail to its head.
    tails, heads = (mesh.incidence[1, 0] * points).batch()[:, inner]           # [inner] Point-, [inner] Point-
    bars = heads - tails                                                       # [inner] Point-
    # The corner across each edge in the face on its right, as its offset square to the bar.
    far = (mesh.reading(mesh.edge_opposites[:, 1]) * points).batch()           # [inner] Point-
    corner = far - tails                                                       # [inner] Point-
    across = corner - bars * ((bars | corner) / (bars | bars))                # [inner] Point-
    # The jump in slope paired with that offset: how much faster the terrain rises across the bar.
    return -(jump & across) / (across.norm() * bars.norm())                    # [..., inner] Scalar+


paraboloid = points.scalar_norm_squared() / 2                                  # Scalar+[vertices]
inner = mesh.interior_edges
lifted_stresses = stresses(mesh, points, paraboloid)                           # [inner] Scalar+
lifted_duals = (slopes(mesh, points, paraboloid) ^ mv.z) * mv.z                # Plane+[faces]


In [ ]:
save_figure(render.draw(mesh, points, paraboloid, lifted_stresses, lifted_duals), "reciprocal");

## 2. Tension and compression

Now let every height breathe a little about the paraboloid, each with its own phase. Where the terrain still bends convex the bar stays in tension, blue; where a vertex rises far enough above its neighbours the terrain creases concave there, the bar goes into compression, red, and its reciprocal edge turns around, folding the reciprocal diagram over itself. Every interior vertex stays in equilibrium throughout, and every reciprocal edge stays across its bar: the heights only ever change how hard each bar pulls or pushes. The boundary vertices are where the truss is held, and the forces there are its loads.

In the notation of graphic statics, the planar truss is the form diagram and the reciprocal diagram the force diagram: each closed polygon of the force diagram is the equilibrium of the joint it is drawn across, and the polyhedral terrain over the form diagram is its Maxwell lift.

In [ ]:
# How far each height breathes about the paraboloid, the seed of the phases, the frames of one breath and
# the duration of each.
amplitude = 0.02
phase_seed = 3
n_frames = 96
duration_ms = 40

In [ ]:
phases = np.random.default_rng(phase_seed).uniform(0, 2 * np.pi, mesh.n_vertices)
times = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)[:, None]
swings = mv.scalar((amplitude * np.sin(times + phases))[..., None]).field()    # [frames] Scalar+[vertices]
breaths = paraboloid + swings                                                  # [frames] Scalar+[vertices]
breath_stresses = stresses(mesh, points, breaths)                              # [frames] Scalar+[inner]
breath_duals = (slopes(mesh, points, breaths) ^ mv.z) * mv.z                   # [frames] Plane+[faces]

In [ ]:
compressed = (breath_stresses < 0).sum(axis=-1)
print(f"bars in compression over the breath: {compressed.min()} to {compressed.max()} of {len(inner)}")
frames = zip(breaths, breath_stresses, breath_duals)
show_animation(render.animate(mesh, points, frames), "reciprocal_breath", duration_ms)

In [ ]:
# checks
def folds(mesh: Mesh, points: Point, heights: Scalar) -> Scalar:
    """The corner across each interior edge on its right lies above the plane of the face on its left by its
    fold, positive where the terrain bends convex."""
    # The plane of the face on the right passes through that corner at its height, so the fold is the jump from
    # the plane on the left to it, read at the corner lifted to unit height.
    jump = (mesh.reading(mesh.interior_edges, 1) * (~mesh.d[1] * slopes(mesh, points, heights))).batch()   # [..., inner] Plane+
    lifted = (mesh.reading(mesh.edge_opposites[:, 1]) * (points + up)).batch()   # [inner] Point-
    return -(jump & lifted)                                                    # [..., inner] Scalar+


# Every bar is in tension exactly where the terrain bends convex, and some bars go into compression.
assert np.array_equal(breath_stresses > 0, folds(mesh, points, breaths) > 0)
assert (breath_stresses < 0).any()
# Every reciprocal edge pairs to nothing with its bar, and every interior vertex is in equilibrium.
bars = (mesh.d[0] * points).batch()[inner]                                     # [inner] Point-
# The jump in slope across each bar is the faces' planes summed back onto their edges, each as its face runs along it.
assert np.abs(((~mesh.d[1] * breath_duals).batch()[:, inner] & bars).to_array()).max() < 1e-10
# The bars meeting at a vertex each pull it along themselves by their stress, summed back onto their ends
# against the way each bar runs.
pulls = (bars * breath_stresses).field()                                       # [frames] Point-[inner]
bar_ends = mesh.reading(inner, 1) * mesh.d[0]                                  # Scalar+[inner <- vertices]
net = -(~bar_ends * pulls).batch()                                             # [frames, vertices] Point-
interior = mesh.interior(0)
assert (net[:, interior].norm() < 1e-10).all()
# On the paraboloid the stresses are the cotangent weights, half the cotangents of the angles facing each edge,
# and the reciprocal vertices the circumcentres.
edges = mesh.boundary[2, 1] * (mesh.d[0] * points)                             # [3] Point-[faces]
tails, heads = mesh.corners[2, 1]
after, before = edges[tails], edges[heads]                                     # [3] Point-[faces], Point-[faces]
cotangents = -(after.dual_inverse() & before) / after.commutator(before).norm()   # [3] Scalar+[faces]
cotangent_weights = (~mesh.incidence[2, 1] * (cotangents / 2)).sum(axis=0)     # Scalar+[edges]
assert np.abs((lifted_stresses - cotangent_weights.batch()[inner]).to_array()).max() < 1e-10
squared_distances = [(lifted_duals.dual() - corner).scalar_norm_squared().to_array() for corner in mesh.incidence[2, 0] * points]
assert np.abs(squared_distances[0] - squared_distances[1]).max() < 1e-10 and np.abs(squared_distances[0] - squared_distances[2]).max() < 1e-10
